# LLM vs. LLM: cloud run of the detection stage (step 06, Colab GPU)

Runs `scripts/06_run_detection.py` for the selected (model, strategy) cells on a Colab GPU and copies the
raw predictions back to Google Drive after every cell. Decision recorded in `docs/QUESTIONS.md` item 11
(2026-10-28): **all evaluation inference for all six detectors runs on Colab GPUs**, the T4 runtime for the
7B to 14B models (`TIER = colab_t4`) and the L4 runtime for gemma-2-27b (`TIER = colab_l4`); the laptop is
used for development and the dev smoke test only. Decoding, prompts, seed and the llama-cpp-python version
are identical to the laptop; only `configs/cloud.yaml` (threads, GPU layers) differs.

**Drive layout expected under `DRIVE_DIR`** (default `MyDrive/llm-thesis`; the processed data are not in git):

```
llm-thesis/
  data/processed/records.jsonl          <- from the laptop, step 03
  data/processed/partitions.json        <- step 04
  data/processed/variants_eval.jsonl    <- step 05, fixed artefact (sha256 checked against reports/variants_eval_fixed_artifacts.json)
  data/processed/variants_dev.jsonl     <- step 05 dev
  .env                                  <- optional, HF_TOKEN=...; the bartowski GGUF repositories are public, so it is not needed
  results/raw/                          <- written by this notebook, one file per cell, synced after every cell and every 10 min
  results/smoke_gpu/                    <- GPU smoke test on the dev sample (cell 7)
  reports/                              <- detection_run_report.json, smoke_gpu_report.json, smoke_backend_agreement.json, models_download_report.json
  logs/                                 <- one log per cell
```

Before running: Runtime > Change runtime type > T4 GPU (or L4 GPU for `colab_l4`). Run the cells top to
bottom. After a disconnect, run the whole notebook again: results are restored from Drive and every cell
resumes at the first missing item. Exception: a model whose results folder was removed from the repository at
the tag and archived under `results/archive/<model>_v*/` (the Mistral v1 cells, `docs/QUESTIONS.md` item 13)
has its Drive copies moved once into `DRIVE_DIR/results/archive/`, so its cells start fresh. Model weights are
downloaded to `/content/models` (outside the repo dir) and survive a re-clone within the session. To rerun one
model only, set `MODELS` (e.g. `mistral-7b`). When a tier is finished, copy `results/raw/` and `reports/` from Drive into
the repository on the laptop and commit them (nothing is pushed from Colab).

In [ ]:
#@title 1. Parameters
TAG = "v0.6.2-detection-ready"      #@param {type:"string"}
REPO_URL = "https://github.com/aliholmes5544/LLM.git"  #@param {type:"string"}
TIER = "colab_t4"                   #@param ["colab_t4", "colab_l4"]
MODELS = ""                         #@param {type:"string"}
STRATEGIES = "zero_shot,few_shot,cot,role"  #@param {type:"string"}
PARTITION = "eval"                  #@param ["eval", "dev"]
DRIVE_DIR = "/content/drive/MyDrive/llm-thesis"  #@param {type:"string"}
RUN_SMOKE = True                    #@param {type:"boolean"}
RUN_EVAL = True                     #@param {type:"boolean"}
REPO_DIR = "/content/llm"
# MODELS empty = every model whose eval_tier (configs/models.yaml) equals TIER; otherwise comma-separated keys.

In [ ]:
#@title 2. Mount Drive, check the GPU and the Drive layout
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
print("python", sys.version.split()[0])
REQUIRED = ["data/processed/records.jsonl", "data/processed/partitions.json",
            "data/processed/variants_eval.jsonl", "data/processed/variants_dev.jsonl"]
missing = [p for p in REQUIRED if not os.path.exists(os.path.join(DRIVE_DIR, p))]
assert not missing, f"missing on Drive under {DRIVE_DIR}: {missing}"
for d in ("results/raw", "results/smoke_gpu", "reports", "logs"):
    os.makedirs(os.path.join(DRIVE_DIR, d), exist_ok=True)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
expected_gpu = {"colab_t4": "T4", "colab_l4": "L4"}[TIER]
assert expected_gpu in gpu, f"tier {TIER} expects a {expected_gpu} GPU, this runtime has {gpu!r} (Runtime > Change runtime type)"
print("GPU ok:", gpu)

In [ ]:
#@title 3. Clone the repository at the tag (model weights live outside the repo dir and survive a re-clone)
import os, subprocess
MODELS_DIR = "/content/models"
os.chdir("/content")                       # never rm -rf the directory we are standing in
if os.path.exists(REPO_DIR):
    subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", "--quiet", "--branch", TAG, "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.symlink(MODELS_DIR, os.path.join(REPO_DIR, "models"))   # configs/models.yaml local_dir: models (gitignored)
os.chdir(REPO_DIR)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
tag = subprocess.run(["git", "describe", "--tags", "--exact-match"], capture_output=True, text=True).stdout.strip()
assert tag == TAG, f"checked out {tag!r}, expected {TAG!r}"
print("repository at", REPO_DIR, "commit", commit, "tag", tag)

In [ ]:
#@title 4. Install the pinned detection requirements with the CUDA build of llama-cpp-python 0.3.36
import os, re, subprocess, sys
nvcc = subprocess.run(["nvcc", "--version"], capture_output=True, text=True).stdout
m = re.search(r"release (\d+)\.(\d+)", nvcc)
cuda_major = int(m.group(1)) if m else 12
index = ("https://abetlen.github.io/llama-cpp-python/whl/cu130" if cuda_major >= 13
         else "https://abetlen.github.io/llama-cpp-python/whl/cu124")
print("CUDA toolkit:", m.group(0) if m else "unknown", "-> wheel index", index)
pip = [sys.executable, "-m", "pip", "install", "-q"]
subprocess.run(pip + ["llama-cpp-python==0.3.36", "--prefer-binary", "--no-cache-dir", "--extra-index-url", index], check=True)
subprocess.run(pip + ["-r", "requirements-detect.txt"], check=True)
import llama_cpp
assert llama_cpp.__version__ == "0.3.36", llama_cpp.__version__
if not llama_cpp.llama_supports_gpu_offload():
    print("the installed wheel has no GPU offload; building from source with GGML_CUDA=on (10 to 20 min)")
    subprocess.run(pip + ["--force-reinstall", "--no-cache-dir", "--no-binary", "llama-cpp-python", "llama-cpp-python==0.3.36"],
                   check=True, env={**os.environ, "CMAKE_ARGS": "-DGGML_CUDA=on"})
    raise SystemExit("built from source: Runtime > Restart session, then run the notebook again from cell 1")
print("llama-cpp-python", llama_cpp.__version__, "GPU offload", llama_cpp.llama_supports_gpu_offload())
print(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
      .replace("\n", " ").split("llama")[0][-200:])

In [ ]:
#@title 5. Pull the processed data from Drive, verify the fixed variants artefact, restore previous results
import hashlib, json, os, shutil, subprocess, time
os.makedirs("data/processed", exist_ok=True)
for p in REQUIRED:
    shutil.copy2(os.path.join(DRIVE_DIR, p), p)
fixed = json.load(open("reports/variants_eval_fixed_artifacts.json"))["files"]["data/processed/variants_eval.jsonl"]
h = hashlib.sha256(open("data/processed/variants_eval.jsonl", "rb").read()).hexdigest()
assert h == fixed["sha256"], f"variants_eval.jsonl sha256 {h} differs from the fixed artefact {fixed['sha256']}"
print("variants_eval.jsonl verified: sha256", h[:16], "...,", fixed["bytes"], "bytes,", fixed["rows"], "rows")
env = os.path.join(DRIVE_DIR, ".env")
if os.path.exists(env):
    shutil.copy2(env, ".env")

REPORT_FILES = ("detection_run_report.json", "smoke_gpu_report.json", "smoke_backend_agreement.json",
                "models_download_report.json")

def sync_to_drive(quiet=False):
    """results/raw, results/smoke_gpu and the step-06 reports -> Drive (rsync keeps files Drive already has)."""
    for d in ("results/raw", "results/smoke_gpu"):
        if os.path.isdir(d):
            subprocess.run(["rsync", "-a", d + "/", os.path.join(DRIVE_DIR, d) + "/"], check=True)
    for f in REPORT_FILES:
        if os.path.exists(os.path.join("reports", f)):
            shutil.copy2(os.path.join("reports", f), os.path.join(DRIVE_DIR, "reports", f))
    if not quiet:
        print(time.strftime("%H:%M:%S"), "synced results and reports to", DRIVE_DIR)

# restore what earlier sessions produced so that every cell resumes, EXCEPT models whose results folder was
# removed from the repository at this tag and archived under results/archive/<model>_v*/ (e.g. the Mistral v1
# cells, QUESTIONS.md item 13): their Drive copies are moved aside once into DRIVE_DIR/results/archive/, so the
# cells start fresh, and the new partial results sync and restore normally after a disconnect.
# The move happens once per archive (marker file on Drive), so results produced afterwards are restored as usual.
RESULT_DIRS = ("results/raw", "results/smoke_gpu")
for name in (sorted(os.listdir("results/archive")) if os.path.isdir("results/archive") else []):
    model = name.split("_v")[0]
    marker = os.path.join(DRIVE_DIR, "results/archive", name, ".moved-from-drive")
    if os.path.exists(marker) or any(os.path.isdir(os.path.join(d, model)) for d in RESULT_DIRS):
        continue
    for d in RESULT_DIRS:
        src = os.path.join(DRIVE_DIR, d, model)
        if os.path.isdir(src):
            dest = os.path.join(DRIVE_DIR, "results/archive", name, os.path.basename(d), model)
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            shutil.move(src, dest)
            print(f"{d}/{model} on Drive predates the archive {name}: moved to {dest}; the {model} cells start fresh")
    os.makedirs(os.path.dirname(marker), exist_ok=True)
    open(marker, "w").write(time.strftime("%Y-%m-%dT%H:%M:%S") + " " + TAG + "\n")
for d in RESULT_DIRS:
    src = os.path.join(DRIVE_DIR, d)
    if os.path.isdir(src):
        os.makedirs(d, exist_ok=True)
        subprocess.run(["rsync", "-a", src + "/", d + "/"], check=True)
for f in REPORT_FILES:
    src = os.path.join(DRIVE_DIR, "reports", f)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join("reports", f))
prev = subprocess.run(["find", "results", "-name", "*.jsonl"], capture_output=True, text=True).stdout.strip()
print("restored from Drive:\n" + (prev or "(no previous results)"))

In [ ]:
#@title 6. Download the GGUF files for the tier and verify their sha256 (configs/models.yaml)
import subprocess, sys, yaml
mcfg = yaml.safe_load(open("configs/models.yaml"))
MODEL_KEYS = [k.strip() for k in MODELS.split(",") if k.strip()] or \
             [k for k, e in mcfg["models"].items() if e["eval_tier"] == TIER]
unknown = [k for k in MODEL_KEYS if k not in mcfg["models"]]
assert not unknown, f"unknown model keys {unknown}"
print("models:", MODEL_KEYS)
subprocess.run([sys.executable, "scripts/06_download_models.py", "--tier", TIER, "--models", ",".join(MODEL_KEYS)], check=True)
# system text delivery check (QUESTIONS.md item 13) for every downloaded model: vocabulary only, a few seconds
subprocess.run([sys.executable, "-m", "unittest", "-v", "tests.test_system_text"], check=True)
sync_to_drive()

In [ ]:
#@title 7. GPU smoke test on the dev sample; agreement with the laptop (CPU) smoke test
import subprocess, sys
if RUN_SMOKE:
    subprocess.run([sys.executable, "-u", "scripts/06_smoke_dev.py", "--tier", TIER, "--backend", "cuda",
                    "--models", ",".join(MODEL_KEYS), "--out-dir", "results/smoke_gpu",
                    "--report", "reports/smoke_gpu_report.json"], check=True)
    # per-cell percent agreement of the extracted labels (and byte-identical raw outputs) with results/smoke_dev/,
    # the committed CPU run; cells without a CPU run (14B, 27B) are simply absent from the comparison
    r = subprocess.run([sys.executable, "scripts/06_compare_smoke.py", "--a", "results/smoke_dev",
                        "--b", "results/smoke_gpu", "--out", "reports/smoke_backend_agreement.json"])
    if r.returncode != 0:
        print("no cell of this tier has a CPU smoke run to compare with")
    sync_to_drive()
else:
    print("RUN_SMOKE is False, skipped")

In [ ]:
#@title 8. Evaluation cells: one resumable job per (model, strategy); synced to Drive every 10 min and after every cell
import datetime, os, subprocess, sys, threading
STRATS = [s.strip() for s in STRATEGIES.split(",") if s.strip()]
if RUN_EVAL:
    for m in MODEL_KEYS:
        for s in STRATS:
            log_path = os.path.join(DRIVE_DIR, "logs", f"{m}_{s}_{PARTITION}.log")
            print(f"=== {datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d %H:%M:%S}Z {m}/{s}/{PARTITION}", flush=True)
            stop = threading.Event()
            ticker = threading.Thread(target=lambda: [sync_to_drive(quiet=True) for _ in iter(lambda: not stop.wait(600), False)],
                                      daemon=True)
            ticker.start()
            skipped = False
            with open(log_path, "a") as lf:
                p = subprocess.Popen([sys.executable, "-u", "scripts/06_run_detection.py", "--model", m, "--strategy", s,
                                      "--partition", PARTITION, "--tier", TIER, "--backend", "cuda"],
                                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
                for line in p.stdout:
                    print(line, end="", flush=True)
                    lf.write(line)
                    skipped = skipped or "already complete" in line
                rc = p.wait()
            stop.set()
            ticker.join(timeout=5)
            if rc == 0 and skipped:
                print(f"{m}/{s}/{PARTITION}: skipped (already complete)", flush=True)
                continue
            sync_to_drive()
            if rc != 0:
                raise SystemExit(f"{m}/{s}/{PARTITION} exited with code {rc}; see {log_path}; fix and run the notebook again (it resumes)")
else:
    print("RUN_EVAL is False, skipped")

In [ ]:
#@title 9. Summary of the cells in reports/detection_run_report.json
import json, os
path = "reports/detection_run_report.json"
if os.path.exists(path):
    rep = json.load(open(path))
    print(f"{'cell':38s} {'done/expected':>14s} {'invalid':>8s} {'median s':>9s} {'trunc':>5s}  backend  gpu")
    for k, c in sorted(rep["cells"].items()):
        print(f"{k:38s} {c['n_done']:>6d}/{c['n_expected']:<7d} {c['invalid_rate']:>8.1%} {c['latency_median_s']:>9.2f} "
              f"{c['truncated_input']:>5d}  {c['backend']:7s}  {c['gpu']}")
    print("forced overwrites:", len(rep.get("forced_overwrites", [])))
print(f"Next: copy {DRIVE_DIR}/results/raw and {DRIVE_DIR}/reports into the repository on the laptop and commit them.")